In [7]:
print("XXX")

XXX


In [8]:
from email.header import decode_header

def decode_mime_header(s):
    """エンコードされたヘッダー（ファイル名など）をデコードする"""
    if not s:
        return ""
    parts = decode_header(s)
    decoded_parts = []
    for payload, charset in parts:
        if isinstance(payload, bytes):
            decoded_parts.append(payload.decode(charset or "utf-8", errors="replace"))
        else:
            decoded_parts.append(payload)
    return "".join(decoded_parts)


In [9]:

def get_attachments(exclude_extensions, message):
    attachments = []

    # is_multipart() が True の場合:
    # メールは「容器」のような状態です。中には「テキスト版の本文」
    # 「HTML版の本文」「画像」「PDF」などがバラバラに入っています。
    # この場合、さらに中身をループで回して、それぞれのパーツを取り
    # 出す必要がある。
    if not message.is_multipart(): return attachments

    for part in message.walk():
        content_disposition = str(part.get("Content-Disposition"))
        if ("attachment" in content_disposition or 
             "inline" in content_disposition):
            
            raw_filename = part.get_filename()
            if not raw_filename: continue

            filename = decode_mime_header(raw_filename)
            
            # 特定のダメな拡張子を持っていないファイルだけを、
            # 添付ファイルとして受け入れる

            if not any(filename.lower().endswith(ext) 
                            for ext in exclude_extensions):
                attachments.append(filename)
    
    return attachments



In [10]:
import mailbox
import csv

def extract_attachments_list(mbox_path, output_csv):
    mbox = mailbox.mbox(mbox_path)
    # 除外したい拡張子のリスト
    exclude_extensions = ['.p7s']
    
    with open(output_csv, 'w', newline='', encoding='utf-8-sig') as f: # Excelで見やすいようutf-8-sig
        writer = csv.writer(f)
        writer.writerow(['Date', 'From', 'Subject', 'Attachment_Files'])

        for message in mbox:
            
            attachments = get_attachments(exclude_extensions, message)
            
            if attachments:
                writer.writerow([
                    decode_mime_header(message['Date']),
                    decode_mime_header(message['From']),
                    decode_mime_header(message['Subject']),
                    ", ".join(attachments)
                ])

    print(f"完了！ {output_csv} を確認してください。")



In [11]:

# 実行
# file = "/home/yutaka/src_p/260506_Google_mail/split_mbox_files/2020.mbox"
# extract_attachments_list(file, 'attachments_list.csv')

In [12]:
import os
import glob

def process_all_mboxes(input_dir, output_dir):
    # 1) output_y フォルダが存在しない場合は作成
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        print(f"Created directory: {output_dir}")

    # 3) 指定ディレクトリ配下の .mbox ファイルをすべて取得
    # 4) 4桁の年.mbox (例: 2020.mbox) にマッチするものを探す
    mbox_files = glob.glob(os.path.join(input_dir, "[0-9][0-9][0-9][0-9].mbox"))

    if not mbox_files:
        print("No matching .mbox files found.")
        return

    for file_path in mbox_files:
        # ファイル名（2020.mboxなど）を取得
        base_name = os.path.basename(file_path)
        
        # 2) csvファイル名を設定（例: 2020.mbox.csv または 2020.csv）
        # ここでは 2020.mbox.csv となるように設定しています
        # csv_filename = f"{base_name}.csv"
        csv_filename = base_name.replace('.mbox', '.csv')
        output_path = os.path.join(output_dir, csv_filename)

        print(f"Processing: {base_name} -> {output_path}")
        
        # 既存の関数を実行
        try:
            extract_attachments_list(file_path, output_path)
        except Exception as e:
            print(f"Error processing {base_name}: {e}")

# --- 実行セクション ---
input_directory = "/home/yutaka/src_p/260506_Google_mail/split_mbox_files"
output_directory = "/home/yutaka/src_p/260507_Google_mail/output_y"
process_all_mboxes(input_directory, output_directory)

Created directory: /home/yutaka/src_p/260507_Google_mail/output_y
Processing: 2021.mbox -> /home/yutaka/src_p/260507_Google_mail/output_y/2021.csv
完了！ /home/yutaka/src_p/260507_Google_mail/output_y/2021.csv を確認してください。
Processing: 2019.mbox -> /home/yutaka/src_p/260507_Google_mail/output_y/2019.csv
完了！ /home/yutaka/src_p/260507_Google_mail/output_y/2019.csv を確認してください。
Processing: 2022.mbox -> /home/yutaka/src_p/260507_Google_mail/output_y/2022.csv
完了！ /home/yutaka/src_p/260507_Google_mail/output_y/2022.csv を確認してください。
Processing: 2026.mbox -> /home/yutaka/src_p/260507_Google_mail/output_y/2026.csv
完了！ /home/yutaka/src_p/260507_Google_mail/output_y/2026.csv を確認してください。
Processing: 2024.mbox -> /home/yutaka/src_p/260507_Google_mail/output_y/2024.csv
完了！ /home/yutaka/src_p/260507_Google_mail/output_y/2024.csv を確認してください。
Processing: 2016.mbox -> /home/yutaka/src_p/260507_Google_mail/output_y/2016.csv
完了！ /home/yutaka/src_p/260507_Google_mail/output_y/2016.csv を確認してください。
Processing: 2023.mbox 